In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(
    y_true: list[str],
    y_pred: list[str],
    labels: list[str],
    experiment: str,
    plots_dir: str | Path,
) -> None:
    """Gera e salva heatmap da matriz de confusão em valores absolutos."""
    
    # Cria a subpasta 'plots' dentro do diretório fornecido
    plots_subdir = Path(plots_dir) / "plots"
    plots_subdir.mkdir(parents=True, exist_ok=True)

    # Mapeamento de pt-br para en-us exclusivo para o plot
    label_map = {"ALTA": "HIGH", "MEDIA": "MEDIUM", "BAIXA": "LOW"}
    display_labels = [label_map.get(lbl, lbl) for lbl in labels]

    # O cálculo do scikit-learn continua usando os labels originais dos dados
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    fig, ax = plt.subplots(figsize=(len(labels) * 2 + 2, len(labels) * 2))
    
    sns.heatmap(
        cm,
        annot=True, 
        fmt="d",          
        cmap="Blues",
        xticklabels=display_labels, 
        yticklabels=display_labels,  
        ax=ax,
        linewidths=0.5, 
        linecolor="grey",
    )
    
    ax.set_xlabel("Predicted Label", fontsize=13)
    ax.set_ylabel("True Label", fontsize=13)
    ax.set_title(f"Confusion Matrix\n{experiment}", fontsize=14, pad=15)

    out = plots_subdir / f"{experiment}_confusion_matrix.png"
    fig.savefig(out, bbox_inches='tight') 
    plt.close(fig)
    

mode = "2class"
model_variants = ["base", "few_shot", "sft", "sft_cot", "grpo_standard", "grpo_distance"]
labels = ["ALTA", "BAIXA"]

# Diretório base para os outputs (CSV ficará aqui, imagens irão para a subpasta 'plots')
output_dir = f"../outputs/confusion_matrix/{mode}/"
os.makedirs(output_dir, exist_ok=True)

for variant in model_variants:
    filename = f"../outputs/results/qwen_{variant}_{mode}_test_predictions.parquet"
    experiment_name = f"qwen_{mode}_{variant}"
    
    # Verifica se o arquivo existe
    if not os.path.exists(filename):
        print(f"⚠️ Arquivo não encontrado: {filename}\n")
        continue
        
    # Carregar os dados
    df = pd.read_parquet(filename)
    
    y_true = df["nivel_classificacao"].tolist()
    y_pred = df["predicted"].tolist()
    
    # Lógica de filtragem
    valid_mask = [p in labels for p in y_pred]
    y_true_f = [t for t, v in zip(y_true, valid_mask) if v]
    y_pred_f = [p for p, v in zip(y_pred, valid_mask) if v]
    
    total_samples = len(y_pred)
    valid_samples = sum(valid_mask)
    valid_rate = (valid_samples / total_samples) * 100 if total_samples > 0 else 0
    
    # Gerar e salvar CSV da matriz de confusão
    cm_array = confusion_matrix(y_true_f, y_pred_f, labels=labels)
    cm_df = pd.DataFrame(cm_array, index=labels, columns=labels)
    cm_df.index.name = 'Real'
    cm_df.columns.name = 'Predicted'
    
    output_csv = os.path.join(output_dir, f"csvs/{experiment_name}_cm.csv")
    cm_df.to_csv(output_csv)
    
    # Exibir no terminal as estatísticas da iteração
    print(f"\n" + "="*60)
    print(f"MODELO: {experiment_name}")
    print(f"Respostas com formato válido: {valid_samples} de {total_samples} ({valid_rate:.2f}%) ({total_samples - valid_samples} não consideradas)")
    
    # Chamada da função para gerar a imagem (.png)
    plot_confusion_matrix(
        y_true=y_true_f, 
        y_pred=y_pred_f, 
        labels=labels, 
        experiment=experiment_name, 
        plots_dir=output_dir
    )
    
    print("="*60)
    print(cm_df.to_markdown())
    print("\n")